In [ ]:
!pip install transformers pillow pandas gradio torch torchvision

In [ ]:
# 2. Clone Repositori Git Anda
!git clone https://github.com/Brian071/Text-to-Image-Retrieval.git

# 3. MASUK KE FOLDER REPOSITORI
%cd Text-to-Image-Retrieval

# 4. Membuat direktori dataset
!mkdir -p dataset

# 5. Mengunduh dataset dari AWS S3 langsung ke dalam folder 'dataset'
print("Mengunduh dataset... (Ini mungkin memakan waktu beberapa menit)")
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar -O dataset/abo-listings.tar
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar -O dataset/abo-images-small.tar

# 6. Mengekstrak isi dataset
print("Mengekstrak dataset...")
!tar -xf dataset/abo-listings.tar -C dataset
!tar -xf dataset/abo-images-small.tar -C dataset

# 7. Bersihkan file .tar untuk menghemat RAM/Disk Colab
!rm dataset/abo-listings.tar
!rm dataset/abo-images-small.tar

# 8. Hapus paket yang berpotensi konflik
!pip uninstall -y transformers torch torchvision numpy scipy pandas seaborn

# 9. Install dependensi secara aman (termasuk Gradio untuk Web UI)
print("Menginstal dependensi...")
!pip install "numpy<2.0.0" scipy pandas seaborn fastapi uvicorn torch torchvision transformers[torch] qdrant-client>=1.10.0 Pillow "opencv-python<4.11.0" optuna segment-anything python-multipart urllib3>=2 matplotlib gradio

# Install Localtunnel (opsional jika Anda butuh port forwarding khusus, Gradio share=True biasanya sudah cukup)
!npm install -g localtunnel

print("✅ PERSIAPAN SELESAI! Silakan jalankan Cell 2.")

fatal: destination path 'Text-to-Image-Retrieval' already exists and is not an empty directory.
/content/Text-to-Image-Retrieval
Mengunduh dataset... (Ini mungkin memakan waktu beberapa menit)
Mengekstrak dataset...
Found existing installation: transformers 5.13.0
Uninstalling transformers-5.13.0:
  Successfully uninstalled transformers-5.13.0
Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: seaborn 0.13.2
Uninstalling seaborn-0.13.2:
  Successfully uni

In [ ]:
# Membuat folder tempat metadata seharusnya berada
!mkdir -p dataset/images/metadata

# Mengunduh file images.csv.gz yang terlupakan
!wget https://amazon-berkeley-objects.s3.amazonaws.com/images/metadata/images.csv.gz -P dataset/images/metadata/

--2026-07-04 09:20:40--  https://amazon-berkeley-objects.s3.amazonaws.com/images/metadata/images.csv.gz
Resolving amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)... 16.15.253.155, 54.231.192.97, 16.15.255.107, ...
Connecting to amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)|16.15.253.155|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6430535 (6.1M) [text/csv]
Saving to: ‘dataset/images/metadata/images.csv.gz.1’

images.csv.gz.1     100%[===================>]   6.13M  28.6MB/s    in 0.2s    

2026-07-04 09:20:40 (28.6 MB/s) - ‘dataset/images/metadata/images.csv.gz.1’ saved [6430535/6430535]



In [ ]:
import pandas as pd
import gzip
import json
import os
import gc
import torch
import gradio as gr
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. PERSIAPAN DATA (KODE BUATAN ANDA)
# ==========================================
print("=== Tahap 1: Memuat dan Merapikan Data ===")
print("Memuat metadata produk...")
data = []
listing_path = 'dataset/listings/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    listing_path = 'dataset/listings/metadata/listings_0.json.gz'

try:
    with gzip.open(listing_path, 'rt') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                data.append(item)
            except:
                continue
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {listing_path}.")

df_listings = pd.DataFrame(data)

def extract_en_text(item_list):
    if isinstance(item_list, list):
        for entry in item_list:
            if isinstance(entry, dict) and entry.get('language_tag', '').startswith('en_'):
                return entry.get('value', '')
        if len(item_list) > 0 and isinstance(item_list[0], dict):
            return item_list[0].get('value', '')
    return None

if not df_listings.empty:
    df_listings['item_name_en'] = df_listings['item_name'].apply(extract_en_text)
    df_listings['category'] = df_listings['product_type'].apply(lambda x: x[0]['value'] if isinstance(x, list) and len(x) > 0 else None)

print("Memuat metadata gambar...")
df_images = pd.read_csv('dataset/images/metadata/images.csv.gz')

df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')

# Smart Path: Antisipasi folder ekstraksi Amazon yang kadang masuk folder 'small'
def get_path(p):
    p1 = 'dataset/images/small/' + str(p)
    p2 = 'dataset/images/' + str(p)
    if os.path.exists(p1): return p1
    return p2

df_merged['image_path'] = df_merged['path'].apply(get_path)

df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={
    'item_id': 'id',
    'item_name_en': 'text'
})

# Filter khusus kategori sepatu dan hapus yang file fisiknya tidak ada
print("Memfilter kategori SHOES...")
df_shoes = df_final[df_final['category'].astype(str).str.upper() == 'SHOES'].copy()
df_shoes = df_shoes[df_shoes['image_path'].apply(os.path.exists)]
print(f"✅ Selesai! Ditemukan {len(df_shoes)} data sepatu valid untuk AI.")

# ==========================================
# 2. SISTEM AI (CLIP MODEL)
# ==========================================
print("\n=== Tahap 2: Ekstraksi AI ===")
MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 64

class CLIPHandler:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Memuat model {MODEL_NAME} di {self.device}...")
        self.model = CLIPModel.from_pretrained(MODEL_NAME).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(MODEL_NAME)

    def get_image_embeddings(self, image_paths):
        all_embeddings = []
        for i in range(0, len(image_paths), BATCH_SIZE):
            batch_paths = image_paths[i:i+BATCH_SIZE]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = self.processor(images=images, return_tensors="pt").to(self.device)

            with torch.no_grad():
                embeds = self.model.get_image_features(**inputs)
                # Standarisasi Output
                if not isinstance(embeds, torch.Tensor):
                    embeds = embeds.image_embeds if hasattr(embeds, 'image_embeds') else embeds[0]
                if len(embeds.shape) == 3: embeds = embeds[:, 0, :]
                if embeds.shape[-1] != 512 and hasattr(self.model, 'visual_projection'):
                    embeds = self.model.visual_projection(embeds)

                embeds = embeds / embeds.norm(dim=-1, keepdim=True)
                all_embeddings.append(embeds.cpu())

            del images, inputs, embeds
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
        return torch.cat(all_embeddings, dim=0).to(self.device)

    def get_text_embedding(self, text):
        inputs = self.processor(text=[text], return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            embeds = self.model.get_text_features(**inputs)
            if not isinstance(embeds, torch.Tensor):
                embeds = embeds.text_embeds if hasattr(embeds, 'text_embeds') else embeds[0]
            if len(embeds.shape) == 3: embeds = embeds[:, 0, :]
            if embeds.shape[-1] != 512 and hasattr(self.model, 'text_projection'):
                embeds = self.model.text_projection(embeds)
            embeds = embeds / embeds.norm(dim=-1, keepdim=True)
        return embeds

# Inisialisasi Model & Ekstrak Vektor Data Sepatu
clip = CLIPHandler()
image_paths = df_shoes['image_path'].tolist()

if image_paths:
    print(f"Mulai menghitung vektor gambar...")
    image_embeddings = clip.get_image_embeddings(image_paths)
    print("✅ Vektor AI berhasil dihitung!")
else:
    image_embeddings = None
    print("❌ Tidak ada gambar untuk diproses.")

# Fungsi Pencarian
def search(query, top_k=4):
    if image_embeddings is None or len(image_paths) == 0:
        return []

    # 1. PROMPT ENGINEERING (Rahasia agar CLIP lebih pintar)
    enhanced_query = f"a clear photo of a {query}, shoe product on a white background"

    text_emb = clip.get_text_embedding(enhanced_query)

    # 2. COSINE SIMILARITY
    similarities = F.cosine_similarity(text_emb, image_embeddings)

    if similarities.dim() == 0: similarities = similarities.unsqueeze(0)
    top_indices = similarities.argsort(descending=True)[:top_k].cpu().numpy()

    return [image_paths[i] for i in top_indices]

# ==========================================
# 3. ANTARMUKA WEB (GRADIO)
# ==========================================
print("\n=== Tahap 3: Membuka Web App ===")
def search_images_ui(query):
    result_paths = search(query, top_k=4)
    results = []
    for p in result_paths:
        row = df_shoes[df_shoes['image_path'] == p]
        if not row.empty:
            # Mengambil deskripsi dari kolom 'text' buatan Anda
            teks = str(row['text'].values[0])
            teks = teks[:90] + "..." if len(teks) > 90 else teks
        else:
            teks = "Deskripsi tidak ditemukan"
        results.append((p, teks))
    return results

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>👟 Shoe Catalog AI Search</h1>")

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(label="Deskripsikan Sepatu yang Dicari", placeholder="Contoh: Red sneakers, black leather boots")
        with gr.Column(scale=1):
            search_btn = gr.Button("🔍 Cari Sepatu", variant="primary")

    gallery = gr.Gallery(label="Hasil Rekomendasi (Visual + Teks)", show_label=True, columns=2, rows=2, object_fit="contain", height=600)

    search_btn.click(fn=search_images_ui, inputs=query_input, outputs=gallery)
    query_input.submit(fn=search_images_ui, inputs=query_input, outputs=gallery)

demo.launch(share=True, debug=True)

=== Tahap 1: Memuat dan Merapikan Data ===
Memuat metadata produk...
Memuat metadata gambar...
Memfilter kategori SHOES...
✅ Selesai! Ditemukan 761 data sepatu valid untuk AI.

=== Tahap 2: Ekstraksi AI ===
Memuat model openai/clip-vit-base-patch32 di cpu...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Mulai menghitung vektor gambar...
✅ Vektor AI berhasil dihitung!

=== Tahap 3: Membuka Web App ===
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c94ea41e505f634a8f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c94ea41e505f634a8f.gradio.live


In [ ]:
import torch.nn.functional as F
import gradio as gr

print(f"🔄 Memastikan {len(df_shoes)} gambar sepatu masuk ke memori...")

# 1. Ekstrak ulang HANYA JIKA memori belum cocok (Biar cepat)
image_paths = df_shoes['image_path'].tolist()
if 'image_embeddings' not in locals() or image_embeddings.shape[0] != len(image_paths):
    print("Menghitung ulang vektor gambar (Mohon tunggu sebentar)...")
    image_embeddings = clip.get_image_embeddings(image_paths)
print("✅ Vektor siap!")

# 2. FUNGSI PENCARIAN YANG BENAR (Tanpa kata "white"!)
def search_accurate(query, top_k=4):
    if image_embeddings is None or len(image_paths) == 0:
        return []

    # Prompt murni, tidak ada kata jebakan
    clean_query = f"a photo of {query}"
    text_emb = clip.get_text_embedding(clean_query)

    # Cosine Similarity bawaan PyTorch
    similarities = F.cosine_similarity(text_emb, image_embeddings)

    if similarities.dim() == 0: similarities = similarities.unsqueeze(0)
    top_indices = similarities.argsort(descending=True)[:top_k].cpu().numpy()

    return [image_paths[i] for i in top_indices]

# 3. RENDER WEB APP
def search_images_ui_fixed(query):
    result_paths = search_accurate(query, top_k=4)
    results = []
    for p in result_paths:
        row = df_shoes[df_shoes['image_path'] == p]
        if not row.empty:
            teks = str(row['text'].values[0])
            teks = teks[:90] + "..." if len(teks) > 90 else teks
        else:
            teks = "Deskripsi tidak ditemukan"
        results.append((p, teks))
    return results

# Tutup UI lama jika masih menyala
try: demo.close()
except: pass

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"<h1 style='text-align: center;'>👟 Shoe Catalog AI Search (FIXED)</h1>")
    gr.Markdown(f"<p style='text-align: center;'>Total Data Tersedia: {len(df_shoes)} Sepatu</p>")

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(label="Cari Sepatu", placeholder="Ketik: Black boots, red heels, gold shoes...")
        with gr.Column(scale=1):
            search_btn = gr.Button("🔍 Cari", variant="primary")

    gallery = gr.Gallery(label="Hasil Pencarian", show_label=True, columns=2, rows=2, object_fit="contain", height=600)

    search_btn.click(fn=search_images_ui_fixed, inputs=query_input, outputs=gallery)
    query_input.submit(fn=search_images_ui_fixed, inputs=query_input, outputs=gallery)

demo.launch(share=True, debug=True)

🔄 Memastikan 761 gambar sepatu masuk ke memori...
✅ Vektor siap!
Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6774803135dc52f3c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
import gradio as gr
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os
import gc

print("🧹 1. Membersihkan sisa memori kotor dari Cell lama...")
for key in ['clip', 'image_embeddings', 'CLIPHandler']:
    if key in globals():
        del globals()[key]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# ==========================================
# A. PERSIAPAN MODEL (STANDAR HUGGINGFACE MURNI)
# ==========================================
print("🚀 2. Memuat Model AI (Tanpa Dropout Bias)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

# ==========================================
# B. EKSTRAKSI VEKTOR YANG AMAN (ANTI ERROR)
# ==========================================
print("⏳ 3. Menghitung ulang vektor gambar dengan dimensi yang benar...")
image_paths = df_shoes['image_path'].tolist()

def get_clean_image_embeddings(paths):
    embeds_list = []
    for i in range(0, len(paths), 64):
        batch = paths[i:i+64]
        imgs = [Image.open(p).convert("RGB") for p in batch]
        inputs = processor(images=imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.get_image_features(**inputs)

            # PERBAIKAN FATAL: Ekstraksi Tensor yang Aman dikembalikan!
            features = out if isinstance(out, torch.Tensor) else (out.image_embeds if hasattr(out, 'image_embeds') else out[0])

            # Antisipasi jika dimensi tidak sesuai
            if len(features.shape) == 3: features = features[:, 0, :]
            if features.shape[-1] != 512 and hasattr(model, 'visual_projection'):
                features = model.visual_projection(features)

            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeds_list.append(features.cpu())
    return torch.cat(embeds_list, dim=0)

image_embeddings_fresh = get_clean_image_embeddings(image_paths)
print("✅ Vektor AI siap digunakan!")

# ==========================================
# C. LOGIKA PENCARIAN (PROMPT ENSEMBLING)
# ==========================================
def search_accurate(query, top_k=4):
    if not query: return [], []

    # 4 Sudut pandang kalimat untuk menjinakkan bias kata "shoes"
    prompts = [
        f"a clear photo of a {query} shoe",
        f"a footwear that is completely {query} in color",
        f"the color {query} on a shoe product",
        f"a photo focusing on the {query} color of the sneakers"
    ]

    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = model.get_text_features(**inputs)

        # PERBAIKAN FATAL: Ekstraksi Tensor Teks yang Aman dikembalikan!
        text_features = out if isinstance(out, torch.Tensor) else (out.text_embeds if hasattr(out, 'text_embeds') else out[0])

        if len(text_features.shape) == 3: text_features = text_features[:, 0, :]
        if text_features.shape[-1] != 512 and hasattr(model, 'text_projection'):
            text_features = model.text_projection(text_features)

        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        mean_text_emb = text_features.mean(dim=0, keepdim=True)
        mean_text_emb = mean_text_emb / mean_text_emb.norm(p=2, dim=-1, keepdim=True)

    similarities = F.cosine_similarity(mean_text_emb.cpu(), image_embeddings_fresh)

    top_indices = similarities.argsort(descending=True)[:top_k].numpy()
    scores = similarities[top_indices].numpy()
    return top_indices, scores

# ==========================================
# D. ANTARMUKA WEB
# ==========================================
try: gr.close_all()
except: pass

with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>👟 Shoe Catalog AI Search (PURE CORE)</h1>")

    with gr.Row():
        with gr.Column(scale=4):
            query_input = gr.Textbox(label="Cari Sepatu (Fokus ke Warna)", placeholder="Ketik: black, red, yellow, dark brown...", lines=1)
        with gr.Column(scale=1):
            search_btn = gr.Button("🔍 Cari", variant="primary")

    image_components, text_components = [], []

    for i in range(2):
        with gr.Row():
            for j in range(2):
                with gr.Column():
                    img = gr.Image(label=f"Rank {i*2+j+1}", type="filepath", height=280)
                    txt = gr.Textbox(label="Informasi Produk", lines=4, interactive=False)
                    image_components.append(img)
                    text_components.append(txt)

    def run_search(query):
        if not query: return [None]*4 + [""]*4
        indices, scores = search_accurate(query, top_k=4)
        out_imgs, out_txts = [], []

        for rank, idx in enumerate(indices):
            p = image_paths[idx]
            score = scores[rank]
            row = df_shoes[df_shoes['image_path'] == p]
            teks_asli = str(row['text'].values[0]) if not row.empty else "Tidak ada teks"
            out_imgs.append(p)
            out_txts.append(f"🔥 Cosine Score: {score:.3f}\n\n📝 Detail:\n{teks_asli}")

        return out_imgs + out_txts

    all_outputs = image_components + text_components
    search_btn.click(fn=run_search, inputs=query_input, outputs=all_outputs)
    query_input.submit(fn=run_search, inputs=query_input, outputs=all_outputs)

demo.launch(share=True, debug=True)

🧹 1. Membersihkan sisa memori kotor dari Cell lama...
🚀 2. Memuat Model AI (Tanpa Dropout Bias)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

⏳ 3. Menghitung ulang vektor gambar dengan dimensi yang benar...
✅ Vektor AI siap digunakan!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d9173323c73adafa65.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
